In [60]:
import pandas as pd
import numpy as np
import matplotlib as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim

import math

In [61]:
# 1. Load the raw data
df = pd.read_csv("/home/dilith_s_b_s/UoP/Sem_4/CO5420/DLProject/beijing-pm25-forecasting/data/raw/train_raw.csv")

# 2. Drop the useless ID index column
if 'No' in df.columns:
    df = df.drop(columns=['No'])

# 3. Recreate the datetime column! (This is what caused your error)
df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])

# Data Preprocessing

## Handling Missing Values

**Objective:** Fill `NaN` values without introducing data leakage.

**Reasoning:** In standard machine learning, we might fill missing data with the overall column mean. In time-series forecasting, this causes *data leakage* by using future information to fill past gaps. Instead, we use a **Forward-Fill (`ffill`)** strategy grouped by the monitoring `station`. This propagates the last known valid sensor reading forward in time. We follow up with a **Backward-Fill (`bfill`)** specifically to catch any missing values at the very beginning of a station's record where forward-fill cannot reach.

In [62]:
# 1. Create a copy to keep our raw data intact
df_clean = df.copy()

# 2. Sort by station and time to ensure forward-fill actually moves forward in time
df_clean = df_clean.sort_values(by=['station', 'datetime'])

# 3. Get all columns that actually have missing values
cols_to_impute = df_clean.columns[df_clean.isnull().any()].tolist()

# 4. Group by station and safely apply forward fill, then backward fill
for col in cols_to_impute:
    # Forward fill: use previous valid hour to fill current missing hour
    df_clean[col] = df_clean.groupby('station')[col].ffill()
    
    # Backward fill: only affects missing values at the very beginning of a station's record
    df_clean[col] = df_clean.groupby('station')[col].bfill()

# 5. Verify that no missing values remain
missing_after = df_clean.isnull().sum().sum()

print("--- Imputation Complete ---")
print(f"Total missing values remaining: {missing_after}")

# Display remaining nulls per column just to be absolutely certain
print("\nMissing values per column:")
print(df_clean.isnull().sum())

--- Imputation Complete ---
Total missing values remaining: 0

Missing values per column:
year        0
month       0
day         0
hour        0
PM2.5       0
PM10        0
SO2         0
NO2         0
CO          0
O3          0
TEMP        0
PRES        0
DEWP        0
RAIN        0
wd          0
WSPM        0
station     0
datetime    0
dtype: int64


## Encoding Catagorical Variables

**Objective:** Convert text-based categorical columns (`wd` and `station`) into a numerical format suitable for neural networks.

**Reasoning:** Neural networks cannot process text strings; they require numbers. We have two main options: Label Encoding (e.g., 1, 2, 3) or One-Hot Encoding (binary 0/1 columns). Because wind direction and station identities have no natural mathematical hierarchy or order (Station B is not mathematically "greater" than Station A), we use **One-Hot Encoding**. This prevents the neural network from learning false mathematical relationships between categories.


In [63]:
# 1. Apply one-hot encoding to 'wd' and 'station'
df_encoded = pd.get_dummies(df_clean, columns=['wd', 'station'], drop_first=False)

# 2. Convert these to integers (1/0) for our neural network.
bool_cols = df_encoded.select_dtypes(include=['bool']).columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

# 3. Print the results to verify
print(f"Old shape (before encoding): {df_clean.shape}")
print(f"New shape (after encoding): {df_encoded.shape}")

# Let's peek at some of the new column names
print("\nNew columns added (sample):")
print([col for col in df_encoded.columns if 'wd_' in col or 'station_' in col][:10], "...")

Old shape (before encoding): (315648, 18)
New shape (after encoding): (315648, 44)

New columns added (sample):
['wd_E', 'wd_ENE', 'wd_ESE', 'wd_N', 'wd_NE', 'wd_NNE', 'wd_NNW', 'wd_NW', 'wd_S', 'wd_SE'] ...


## Temporal validation Split

**Objective:** Divide the dataset into training and validation sets to evaluate model performance accurately.

**Reasoning:** Using a standard random `train_test_split` on time-series data destroys the temporal order and causes severe data leakage (e.g., using a row from 5:00 PM to predict a row at 4:00 PM). Instead, we perform a **chronological split**. We use the first two years (March 2013 – Feb 2015) for training and the final year (March 2015 – Feb 2016) for validation. This ensures our validation set contains a complete seasonal cycle—including extreme winter smog events—providing a realistic evaluation of our model.


In [64]:
# 1. Ensure chronological order
df_encoded = df_encoded.sort_values('datetime')

# 2. Find the start and end dates
start_date = df_encoded['datetime'].min()
end_date = df_encoded['datetime'].max()

print(f"Dataset starts on: {start_date}")
print(f"Dataset ends on: {end_date}")

# 3. Define the cutoff date string
cutoff_date = '2015-03-01 00:00:00'

# 4. Split the data based on the datetime column
# Use strictly less than (<) for train, and greater than or equal to (>=) for val
train_df = df_encoded[df_encoded['datetime'] < cutoff_date].copy()
val_df = df_encoded[df_encoded['datetime'] >= cutoff_date].copy()

# 5. Print the shapes
print(f"Training data shape: {train_df.shape}")
print(f"Validation data shape: {val_df.shape}")

Dataset starts on: 2013-03-01 00:00:00
Dataset ends on: 2016-02-29 23:00:00
Training data shape: (210240, 44)
Validation data shape: (105408, 44)


## Feature Scaling (Standardization)

**Objective:** Scale continuous features to have a mean of 0 and a standard deviation of 1.

**Reasoning:** Neural networks struggle when features have vastly different magnitudes (e.g., Atmospheric Pressure is ~1000, while Wind Speed is ~2). Without scaling, the network might assume Pressure is hundreds of times more important simply because the number is larger. 
**The Golden Rule:** To prevent data leakage, we `fit` the scaler (calculate the mean and standard deviation) **strictly on the training data**. We then use those exact parameters to `transform` both the training and validation datasets, ensuring no future information influences the scaling.

In [ ]:
# 1. Define the continuous columns to scale
cols_to_scale = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']

# 2. Initialize the scaler
scaler = StandardScaler()

# 3. Fit on training data AND transform training data
train_df[cols_to_scale] = scaler.fit_transform(train_df[cols_to_scale])

# 4. Transform validation data using the same scaler (NO fitting here!)
val_df[cols_to_scale] = scaler.transform(val_df[cols_to_scale])

# 5. Verify the scaling worked (Mean should be ~0, Std should be ~1 for train_df)
print("Training Data PM2.5 Mean after scaling:", train_df['PM2.5'].mean())
print("Training Data PM2.5 Std after scaling:", train_df['PM2.5'].std())

Training Data PM2.5 Mean after scaling: -6.056376438746668e-17
Training Data PM2.5 Std after scaling: 1.0000023782428829


## Time Feature Extraction (Cyclical Encoding)

**Objective:** Encode time variables (hour, month) so the model understands their cyclical nature.

**Reasoning:** If we feed raw hours (0-23) into a neural network, it assumes a massive jump between hour 23 and hour 0. By applying sine and cosine transformations, we map these time units onto a circle. This preserves the true temporal distance (e.g., December is physically close to January, and 11:00 PM is close to Midnight), allowing the model to learn daily and seasonal periodicities smoothly.

In [66]:
# Apply to both train and val sets
for d in [train_df, val_df]:
    # Extract raw hour and month (if not already present as integers)
    hour = d['datetime'].dt.hour
    month = d['datetime'].dt.month
    
    # Cyclical encoding for hour (24 hours)
    d['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    d['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    
    # Cyclical encoding for month (12 months)
    d['month_sin'] = np.sin(2 * np.pi * month / 12)
    d['month_cos'] = np.cos(2 * np.pi * month / 12)

print("Cyclical time features added!")
print(f"New training shape: {train_df.shape}")

Cyclical time features added!
New training shape: (210240, 48)


## Baseline Model (Naive Persistence)

**Objective:** Establish a performance benchmark using a simple heuristic before training complex deep learning models.

**Reasoning:** In time-series forecasting, a model is only valuable if it can beat simple heuristics. The most common baseline is the **Naive Persistence Model**, which assumes that the value at time $t+1$ will be equal to the value at time $t$ ($\hat{y}_{t+1} = y_t$). We calculate the Root Mean Squared Error (RMSE) of this baseline on our unscaled validation set. Any deep learning architecture we build later **must** achieve a lower RMSE than this baseline to be considered successful.

In [67]:
# 1. Get the unscaled validation data using the same cutoff date
val_clean = df_clean[df_clean['datetime'] >= '2015-03-01'].copy()

# 2. Shift the PM2.5 column by 1 to simulate "predicting the next hour using the current hour"
# We must group by station so we don't accidentally predict Station B's first hour using Station A's last hour!
val_clean['baseline_pred'] = val_clean.groupby('station')['PM2.5'].shift(1)

# 3. Drop the first row of each station (since it has no prior hour to pull a prediction from)
val_clean = val_clean.dropna(subset=['baseline_pred'])

# 4. Calculate RMSE
rmse_baseline = np.sqrt(mean_squared_error(val_clean['PM2.5'], val_clean['baseline_pred']))

print(f"Naive Baseline RMSE: {rmse_baseline:.4f}")

Naive Baseline RMSE: 19.8542


## Deep Learning Preparation (Sliding Windows)

**Objective:** Transform 2D continuous tabular data into 3D tensors for deep learning sequence models.

**Reasoning:** Temporal neural networks (RNNs, LSTMs) expect input data in the shape of `(Batch_Size, Sequence_Length, Num_Features)`. Our competition requires predicting the 1-hour ahead PM2.5 concentration using the preceding 24 hours. Therefore, our `Sequence_Length` is 24. We must use a **Sliding Window** approach: 
- Window 1: Hours 0-23 $\rightarrow$ Predicts Hour 24
- Window 2: Hours 1-24 $\rightarrow$ Predicts Hour 25
We must isolate this process per monitoring station to ensure we do not accidentally create a 24-hour sequence that contains 12 hours from Station A and 12 hours from Station B.

In [ ]:
def create_windows(df, window_size=24, target_col='PM2.5'):
    # Find all the one-hot encoded station columns
    station_cols = [c for c in df.columns if 'station_' in c]
    
    X, y = [], []
    
    # Iterate through each station separately
    for st_col in station_cols:
        # Isolate the data for this specific station and ensure chronological order
        st_data = df[df[st_col] == 1].sort_values('datetime')
        
        # --- FIX: Drop datetime AND the raw unscaled time columns ---
        cols_to_drop = ['datetime', 'year', 'month', 'day', 'hour']
        features = st_data.drop(columns=cols_to_drop).values
        
        # Find the new index of our target variable (which will now be 0)
        target_idx = st_data.drop(columns=cols_to_drop).columns.get_loc(target_col)
        
        # Slide the window
        for i in range(len(features) - window_size):
            X.append(features[i : i + window_size, :])
            y.append(features[i + window_size, target_idx]) # Predict the NEXT hour
            
    return np.array(X), np.array(y)

# Generate the windows
print("Generating training windows...")
X_train, y_train = create_windows(train_df)

print("Generating validation windows...")
X_val, y_val = create_windows(val_df)

print("\n--- Tensor Shapes ---")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")

Generating training windows...
Generating validation windows...

--- Tensor Shapes ---
X_train shape: (209952, 24, 43)
y_train shape: (209952,)
X_val shape: (105120, 24, 43)
y_val shape: (105120,)


## PyTorch DataLoaders

**Objective:** Convert NumPy arrays into PyTorch Tensors and wrap them in DataLoaders for memory-efficient batching.

**Reasoning:** Deep learning models train iteratively using Gradient Descent. Passing the entire dataset into the model at once requires too much memory and leads to poor generalization. By wrapping our tensors in a PyTorch `DataLoader`, we can feed the data into the model in smaller `batches` (e.g., 256 windows at a time). 
- **Training DataLoader:** We shuffle the windows (`shuffle=True`) to prevent the model from memorizing the chronological order of the dataset, forcing it to learn the actual feature-to-target relationships. (Note: The 24-hour temporal order *inside* each window remains strictly intact).
- **Validation DataLoader:** We do not shuffle (`shuffle=False`) so we can evaluate the model sequentially and plot realistic prediction graphs later.

In [69]:
# 1. Convert to PyTorch Tensors (Float32)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

# 2. Create Datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

# 3. Create DataLoaders
BATCH_SIZE = 256

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 4. Verify the batch shape
for X_batch, y_batch in train_loader:
    print(f"Batch X shape: {X_batch.shape}")
    print(f"Batch y shape: {y_batch.shape}")
    break # We only want to see the first batch

Batch X shape: torch.Size([256, 24, 43])
Batch y shape: torch.Size([256])


## Model Development - Advanced Architecture (Residual LSTM)

**Objective:** Implement a skip connection so the LSTM predicts the *change* in pollution rather than the absolute value.

**Reasoning:** Air pollution is highly autocorrelated (the PM2.5 level right now is the strongest predictor of the PM2.5 level in one hour). Standard LSTMs struggle to retain this exact scalar value across complex gating mechanisms. By using a **Skip Connection**, we bypass the LSTM and send the PM2.5 value from the 24th hour straight to the output. The LSTM is then tasked *only* with predicting the **Residual** ($\Delta$)—how much the weather will cause the pollution to go up or down in the next hour.

In [72]:
# 1. Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Training on: {device}")

# 2. Define the corrected Model (input_size=43)
class ResidualLSTM(nn.Module):
    def __init__(self, input_size=43, hidden_size=64, num_layers=2, dropout=0.2):
        super(ResidualLSTM, self).__init__()
        
        self.lstm = nn.LSTM(input_size=input_size, 
                            hidden_size=hidden_size, 
                            num_layers=num_layers, 
                            batch_first=True,
                            dropout=dropout)
        
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        # 1. Extract the Naive Baseline (Last hour's PM2.5, which is now safely feature index 0)
        last_pm25 = x[:, -1, 0]
        
        # 2. Run the sequence through the LSTM
        out, _ = self.lstm(x)
        last_time_step = out[:, -1, :]
        
        # 3. Predict the RESIDUAL (the change)
        delta = self.fc(last_time_step).squeeze(1)
        
        # 4. Final Prediction = Baseline + Predicted Change
        return last_pm25 + delta

# Initialize the new model
model = ResidualLSTM().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# 3. Training Loop
EPOCHS = 10

for epoch in range(EPOCHS):
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        
        optimizer.zero_grad() 
        loss.backward()       
        optimizer.step()      
        
        train_loss += loss.item() * X_batch.size(0)
        
    train_loss /= len(train_loader.dataset)
    
    # --- VALIDATION PHASE ---
    model.eval() 
    val_loss = 0.0
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item() * X_batch.size(0)
            
    val_loss /= len(val_loader.dataset)
    val_rmse = math.sqrt(val_loss) 
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f} | Val RMSE: {val_rmse:.4f}")

# --- 4. INVERSE TRANSFORM (UNSCALING) ---
all_preds = []
all_trues = []

model.eval()
with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch).cpu().numpy()
        all_preds.extend(preds)
        all_trues.extend(y_batch.cpu().numpy())

all_preds = np.array(all_preds)
all_trues = np.array(all_trues)

dummy_preds = np.zeros((len(all_preds), 11))
dummy_preds[:, 0] = all_preds
unscaled_preds = scaler.inverse_transform(dummy_preds)[:, 0]

dummy_trues = np.zeros((len(all_trues), 11))
dummy_trues[:, 0] = all_trues
unscaled_trues = scaler.inverse_transform(dummy_trues)[:, 0]

real_rmse = math.sqrt(mean_squared_error(unscaled_trues, unscaled_preds))

print(f"\n--- FINAL RESULTS ---")
print(f"True Validation RMSE: {real_rmse:.4f}")
print(f"Baseline to beat: 19.8542")

Training on: cuda
Epoch [1/10] | Train MSE: 0.0597 | Val MSE: 0.0506 | Val RMSE: 0.2250
Epoch [2/10] | Train MSE: 0.0534 | Val MSE: 0.0492 | Val RMSE: 0.2218
Epoch [3/10] | Train MSE: 0.0510 | Val MSE: 0.0502 | Val RMSE: 0.2240
Epoch [4/10] | Train MSE: 0.0497 | Val MSE: 0.0514 | Val RMSE: 0.2267
Epoch [5/10] | Train MSE: 0.0486 | Val MSE: 0.0501 | Val RMSE: 0.2239
Epoch [6/10] | Train MSE: 0.0477 | Val MSE: 0.0498 | Val RMSE: 0.2232
Epoch [7/10] | Train MSE: 0.0468 | Val MSE: 0.0497 | Val RMSE: 0.2230
Epoch [8/10] | Train MSE: 0.0458 | Val MSE: 0.0512 | Val RMSE: 0.2262
Epoch [9/10] | Train MSE: 0.0451 | Val MSE: 0.0511 | Val RMSE: 0.2259
Epoch [10/10] | Train MSE: 0.0445 | Val MSE: 0.0514 | Val RMSE: 0.2267

--- FINAL RESULTS ---
True Validation RMSE: 18.0926
Baseline to beat: 19.8542


## Final Pipeline

**Objective:** Process the unseen test data using our exact training pipeline and generate the final `submission.csv`.

**Reasoning:** To prevent data leakage and ensure our neural network understands the test data, we must apply the *exact same transformations* used during training. This means using `.transform()` instead of `.fit_transform()` for our scaler, and ensuring our one-hot encoded columns perfectly match the training set's shape. Finally, we run our sliding window function, pass the tensors through our trained `ResidualLSTM`, unscale the predictions, and format them into the specific CSV structure needed.

In [78]:
# 1. Load the flat test data
test_df = pd.read_csv("/home/dilith_s_b_s/UoP/Sem_4/CO5420/DLProject/beijing-pm25-forecasting/data/raw/test.csv")

# 2. UNFLATTEN THE DATA
print("Unflattening test data...")
base_cols = ['year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM']
long_rows = []

for _, row in test_df.iterrows():
    seq_id = row['id']
    station = row['station']
    # Iterate backwards from 24 to 1 to maintain chronological time order
    for lag in range(24, 0, -1): 
        hour_data = {'id': seq_id, 'station': station}
        for col in base_cols:
            hour_data[col] = row[f'{col}_lag_{lag}']
        long_rows.append(hour_data)

test_long = pd.DataFrame(long_rows)

# 3. IMPUTATION (Grouped strictly by sequence ID so we don't leak between different test cases)
print("Imputing missing values...")
cols_to_impute = test_long.columns[test_long.isnull().any()].tolist()
for col in cols_to_impute:
    test_long[col] = test_long.groupby('id')[col].ffill().bfill()

# 4. DATETIME & CYCLICAL ENCODING
test_long['datetime'] = pd.to_datetime(test_long[['year', 'month', 'day', 'hour']])
test_long['hour_sin'] = np.sin(2 * np.pi * test_long['hour'] / 24)
test_long['hour_cos'] = np.cos(2 * np.pi * test_long['hour'] / 24)
test_long['month_sin'] = np.sin(2 * np.pi * test_long['month'] / 12)
test_long['month_cos'] = np.cos(2 * np.pi * test_long['month'] / 12)

# 5. SCALING (Use the pre-fit scaler from training)
cols_to_scale = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
test_long[cols_to_scale] = scaler.transform(test_long[cols_to_scale])

# 6. ONE-HOT ENCODING & COLUMN ALIGNMENT
test_encoded = pd.get_dummies(test_long, columns=['wd', 'station'], drop_first=False)
bool_cols = test_encoded.select_dtypes(include=['bool']).columns
test_encoded[bool_cols] = test_encoded[bool_cols].astype(int)

# Extract the exact feature list the model expects (from our train_df)
expected_features = train_df.drop(columns=['datetime', 'year', 'month', 'day', 'hour']).columns.tolist()

# Add any missing one-hot columns (e.g., if a specific wind direction didn't appear in the test set)
for c in expected_features:
    if c not in test_encoded.columns:
        test_encoded[c] = 0

# 7. RESHAPE INTO 3D TENSOR
# Isolate just the features, perfectly ordered
X_test_flat = test_encoded[expected_features].values 
# Reshape from (98472, 43) -> (4103, 24, 43)
X_test = X_test_flat.reshape(len(test_df), 24, len(expected_features))
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

print(f"Final Test Tensor Shape: {X_test_tensor.shape}")

# 8. INFERENCE
model.eval()
test_preds = []

with torch.no_grad():
    for i in range(0, len(X_test_tensor), 256):
        batch = X_test_tensor[i : i+256]
        preds = model(batch).cpu().numpy()
        test_preds.extend(preds)

test_preds = np.array(test_preds)

# 9. UNSCALE PREDICTIONS (Dummy Matrix Trick)
dummy_test_preds = np.zeros((len(test_preds), 11))
dummy_test_preds[:, 0] = test_preds # PM2.5 is at index 0
unscaled_test_preds = scaler.inverse_transform(dummy_test_preds)[:, 0]

# 10. CREATE SUBMISSION
submission = pd.DataFrame({
    'id': test_df['id'], 
    'PM2.5': unscaled_test_preds
})

submission.to_csv("submission.csv", index=False)
print("\n--- Pipeline Complete ---")
print("Saved to submission.csv! Ready for Kaggle upload.")

Unflattening test data...
Imputing missing values...
Final Test Tensor Shape: torch.Size([4103, 24, 43])

--- Pipeline Complete ---
Saved to submission.csv! Ready for Kaggle upload.
